In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Install YOLO
!pip install -q ultralytics pyyaml

from ultralytics import YOLO
from pathlib import Path
import yaml

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
PROJECT_DIR = Path("/content/drive/MyDrive/Machine_Learning")

DATASET_DIR = PROJECT_DIR / "Dataset/YOLO_Dataset/multi_class_road_crossing_1280_yolov8"

MODEL_FILE = PROJECT_DIR / "Models" / "yolov8m.pt"
OUTPUT_DIR = PROJECT_DIR / "Ablation_Study/Experiments/multi_yolov8m_1280"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
yaml_dict = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["person"]
}

yaml_path = DATASET_DIR / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(yaml_dict, f)

In [4]:
last_checkpoint = OUTPUT_DIR / "train" / "weights" / "last.pt"

if last_checkpoint.exists():
    print("Resuming training from:", last_checkpoint)
    model = YOLO(str(last_checkpoint))
    resume = True
else:
    print("Starting new training...")
    model = YOLO(str(MODEL_FILE))
    resume = False

Resuming training from: /content/drive/MyDrive/Machine_Learning/Ablation_Study/Experiments/multi_yolov8m_1280/train/weights/last.pt


In [5]:
model.train(
    data=str(yaml_path),

    epochs=50,
    imgsz=1280,
    batch=8,

    optimizer="AdamW",
    lr0=5e-4,
    lrf=0.01,
    weight_decay=5e-4,

    dropout=0.05,
    label_smoothing=0.05,
    patience=10,
    cos_lr=True,
    cache=True,
    workers=2,
    amp=True,
    seed=42,

    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    translate=0.1,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    project=str(OUTPUT_DIR),
    name="train",
    exist_ok=True,

    resume=resume,

    save=True,
    save_period=5,
    plots=True
)

best_model = OUTPUT_DIR / "train" / "weights" / "best.pt"

print("\nLoading Best Model...")
model = YOLO(str(best_model))

WARNING ⚠️ model '/content/drive/MyDrive/Machine_Learning/Ablation_Study/Experiments/multi_yolov8m_1280/train/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/Machine_Learning/Dataset/YOLO_Dataset/multi_class_road_crossing_1280_yolov8/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=Fa

In [6]:
print("\nValidation Results")
metrics = model.val(
    data=str(yaml_path),
    split="val"
)

print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")



Validation Results
Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.3±1.3 ms, read: 0.4±0.1 MB/s, size: 240.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/.shortcut-targets-by-id/1v-Gq4tBK6j-LUvTfFIBTi2oS67OiXVxk/Machine_Learning/Dataset/YOLO_Dataset/multi_class_road_crossing_1280_yolov8/valid/labels.cache... 68 images, 2 backgrounds, 60 corrupt: 100% ━━━━━━━━━━━━ 68/68 20.4Mit/s 0.0s
val: /content/drive/.shortcut-targets-by-id/1v-Gq4tBK6j-LUvTfFIBTi2oS67OiXVxk/Machine_Learning/Dataset/YOLO_Dataset/multi_class_road_crossing_1280_yolov8/valid/images/IMG-20260412-WA0004-1-_jpg.rf.7e3794cf8971095903acd26098440e78.jpg: ignoring corrupt image/label: Label class 2 exceeds dataset class count 1. Possib

In [7]:
print("\nTesting on Test Set")
test_metrics = model.val(
    data=str(yaml_path),
    split="test"
)

print(f"Test Precision : {test_metrics.box.mp:.4f}")
print(f"Test Recall    : {test_metrics.box.mr:.4f}")
print(f"Test mAP@50    : {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95  : {test_metrics.box.map:.4f}")


Testing on Test Set
Ultralytics 8.4.105 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.2 ms, read: 0.2±0.0 MB/s, size: 155.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/.shortcut-targets-by-id/1v-Gq4tBK6j-LUvTfFIBTi2oS67OiXVxk/Machine_Learning/Dataset/YOLO_Dataset/multi_class_road_crossing_1280_yolov8/test/labels... 68 images, 4 backgrounds, 54 corrupt: 100% ━━━━━━━━━━━━ 68/68 1.2s/it 1:22
val: /content/drive/.shortcut-targets-by-id/1v-Gq4tBK6j-LUvTfFIBTi2oS67OiXVxk/Machine_Learning/Dataset/YOLO_Dataset/multi_class_road_crossing_1280_yolov8/test/images/0015_jpg.rf.35c77c5a88747e541b093360252a9e54.jpg: ignoring corrupt image/label: Label class 2 exceeds dataset class count 1. Possible class labels are 0-0
val: /content/drive/.shortcut-targets-by-id/1v-Gq4tBK6j-LUvTfFIBTi2oS67OiXVxk/Machine

In [8]:
import pandas as pd

metrics_data = {
    'Metric': ['Precision', 'Recall', 'mAP@50', 'mAP50-95'],
    'Validation Set': [metrics.box.mp, metrics.box.mr, metrics.box.map50, metrics.box.map],
    'Test Set': [test_metrics.box.mp, test_metrics.box.mr, test_metrics.box.map50, test_metrics.box.map]
}

metrics_df = pd.DataFrame(metrics_data)

# Format the numerical columns to 4 decimal places
metrics_df['Validation Set'] = metrics_df['Validation Set'].apply(lambda x: f'{x:.4f}')
metrics_df['Test Set'] = metrics_df['Test Set'].apply(lambda x: f'{x:.4f}')

# Display the DataFrame as a markdown table
print(metrics_df.to_markdown(index=False))

| Metric    |   Validation Set |   Test Set |
|:----------|-----------------:|-----------:|
| Precision |           0.9227 |     0.8327 |
| Recall    |           0.8889 |     0.7407 |
| mAP@50    |           0.908  |     0.8049 |
| mAP50-95  |           0.605  |     0.4932 |
